# Chapter 04. 수동 에이전트 루프

## 학습 목표
- CausalAgent 클래스를 설계하고 구현한다
- 순차 파이프라인 모드와 자율 에이전트 모드를 비교한다

## 환경 설정

In [1]:
import os, json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any, Callable

# 환경 변수 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("환경 설정 완료!")

환경 설정 완료!


## 1. Agent Excutor 에이전트 루프의 원리

내부적으로 수행하는 작업을 직접 구현한다. 핵심은 while 루프이다.

In [2]:
# ============================================================
# 가장 단순한 수동 에이전트 루프
# ============================================================

def simple_agent_loop(client, messages, tools, tool_registry, max_steps=5):
    """가장 기본적인 에이전트 루프 구현이다."""
    
    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")
        
        # 1) LLM 호출
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_msg = response.choices[0].message
        
        # 2) 도구 호출이 없으면 최종 답변
        if not assistant_msg.tool_calls:
            print(f"[최종 답변] {assistant_msg.content[:200]}...")
            return assistant_msg.content
        
        # 3) assistant 메시지 추가 (tool_calls 포함)
        messages.append(assistant_msg)
        
        # 4) 각 도구 실행 + 결과 주입
        for tc in assistant_msg.tool_calls:
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)
            print(f"[도구 호출] {func_name}({list(func_args.keys())})")
            
            # 도구 실행
            try:
                result = tool_registry[func_name](**func_args)
            except Exception as e:
                result = {"error": str(e)}
            
            # 결과를 tool 메시지로 주입
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False, default=str)
            })
    
    return "최대 반복 횟수에 도달했다."

print("simple_agent_loop() 함수 정의 완료!")
print("→ 이것이 AgentExecutor의 핵심 로직이다.")

simple_agent_loop() 함수 정의 완료!
→ 이것이 AgentExecutor의 핵심 로직이다.


## 간단한 도구와 함께 테스트

In [3]:
# ============================================================
# 간단한 도구를 등록하고 에이전트 루프를 테스트한다
# ============================================================

def get_data_columns(dataset_path: str) -> dict:
    """데이터셋의 컬럼 목록을 반환한다."""
    df = pd.read_csv(dataset_path)
    return {"columns": list(df.columns), "num_rows": len(df), "num_cols": len(df.columns)}

def compute_mean_diff(dataset_path: str, treatment: str, outcome: str) -> dict:
    """처치군/대조군 평균 차이를 계산한다."""
    df = pd.read_csv(dataset_path)
    treated = df[df[treatment] == 1][outcome]
    control = df[df[treatment] == 0][outcome]
    return {"ate": float(treated.mean() - control.mean()), "n_treated": len(treated), "n_control": len(control)}

# 도구 정의
test_tools = [
    {"type": "function", "function": {
        "name": "get_data_columns",
        "description": "데이터셋의 컬럼 목록과 크기를 반환한다.",
        "parameters": {"type": "object", "properties": {
            "dataset_path": {"type": "string"}
        }, "required": ["dataset_path"]}
    }},
    {"type": "function", "function": {
        "name": "compute_mean_diff",
        "description": "이진 처치변수의 처치군/대조군 평균 차이를 계산한다.",
        "parameters": {"type": "object", "properties": {
            "dataset_path": {"type": "string"},
            "treatment": {"type": "string"},
            "outcome": {"type": "string"}
        }, "required": ["dataset_path", "treatment", "outcome"]}
    }}
]

test_registry = {
    "get_data_columns": get_data_columns,
    "compute_mean_diff": compute_mean_diff
}

# 에이전트 루프 실행
messages = [
    {"role": "system", "content": "너는 인과추론 에이전트이다. 도구를 활용하여 분석한다."},
    {"role": "user", "content": "./dataset/online_classroom.csv에서 format_ol이 falsexam에 미치는 효과를 분석해줘."}
]

# result = simple_agent_loop(client, messages, test_tools, test_registry)
print("에이전트 루프 실행 준비 완료!")
print("(실제 데이터 경로에 맞춰 실행하면 된다)")

에이전트 루프 실행 준비 완료!
(실제 데이터 경로에 맞춰 실행하면 된다)


## 2. CausalAgent 클래스 구현

In [5]:
class CausalAgent:
    """OpenAI SDK 기반 인과추론 에이전트이다.
    
    두 가지 실행 모드를 지원한다:
    - run(): 자율 에이전트 모드 (LLM이 도구 선택)
    - run_pipeline(): 순차 파이프라인 모드 (8단계 고정)
    """
    
    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
        self.tools_schema = []      # JSON Schema 리스트
        self.tools_registry = {}    # 이름 → 함수 매핑
        self.messages = []          # 대화 히스토리 (메모리)
        self.system_prompt = """너는 인과추론 전문 에이전트이다.
            사용자의 질문을 분석하고, 적절한 도구를 활용하여 인과효과를 추정한다.
            분석 절차: 데이터 분석 → 변수 식별 → 방법론 선택 → 효과 추정 → 결과 해석"""
    
    def register_tool(self, name: str, func: Callable, schema: dict):
        """도구를 에이전트에 등록한다."""
        self.tools_registry[name] = func
        self.tools_schema.append(schema)
        
    def _execute_tool_calls(self, tool_calls) -> list:
        """도구를 실행하고 결과 메시지를 생성한다."""
        tool_messages = []
        for tc in tool_calls:
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)
            
            print(f"  [실행] {func_name}")
            
            try:
                if func_name in self.tools_registry:
                    result = self.tools_registry[func_name](**func_args)
                else:
                    result = {"error": f"알 수 없는 도구: {func_name}"}
            except Exception as e:
                result = {"error": f"도구 실행 실패: {str(e)}"}
            
            tool_messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False, default=str)
            })
        
        return tool_messages
    
    def run(self, query: str, max_steps: int = 10) -> str:
        """자율 에이전트 모드: LLM이 도구를 자유롭게 선택한다."""
        self.messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": query}
        ]
        
        for step in range(max_steps):
            print(f"\n=== Step {step + 1}/{max_steps} ===")
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=self.tools_schema if self.tools_schema else None,
                tool_choice="auto" if self.tools_schema else None
            )
            
            msg = response.choices[0].message
            
            if not msg.tool_calls:
                self.messages.append({"role": "assistant", "content": msg.content})
                return msg.content
            
            # assistant 메시지 추가
            self.messages.append(msg)
            
            # 도구 실행 + 결과 주입
            tool_results = self._execute_tool_calls(msg.tool_calls)
            self.messages.extend(tool_results)
        
        return "최대 반복 횟수에 도달했다."
    
    def run_pipeline(self, query: str, dataset_path: str, steps: list = None) -> dict:
        """순차 파이프라인 모드: CAIS의 run_causal_analysis()와 동일한 패턴이다.
        
        고정된 순서로 도구를 실행한다.
        이것이 CAIS agent.py의 실제 동작 방식이다.
        """
        if steps is None:
            steps = list(self.tools_registry.keys())
        
        results = {}
        context = {"query": query, "dataset_path": dataset_path}
        
        for i, step_name in enumerate(steps):
            print(f"\n[Pipeline Step {i+1}/{len(steps)}] {step_name}")
            
            if step_name in self.tools_registry:
                try:
                    func = self.tools_registry[step_name]
                    result = func(**context)
                    results[step_name] = result
                    # 다음 단계를 위해 컨텍스트 업데이트
                    if isinstance(result, dict):
                        context.update(result)
                    print(f"  ✓ 완료")
                except Exception as e:
                    print(f"  ✗ 실패: {e}")
                    results[step_name] = {"error": str(e)}
            else:
                print(f"  ⚠ 도구 미등록: {step_name}")
        
        return results
    
    def get_memory(self) -> list:
        """현재 대화 히스토리를 반환한다."""
        return self.messages.copy()
    
    def clear_memory(self):
        """대화 히스토리를 초기화한다."""
        self.messages = []

print("CausalAgent 클래스 정의 완료!")
print("메서드: run() (자율 모드), run_pipeline() (순차 모드)")

CausalAgent 클래스 정의 완료!
메서드: run() (자율 모드), run_pipeline() (순차 모드)


## CausalAgent 테스트

In [7]:
# ============================================================
# CausalAgent 자율 모드 테스트
# ============================================================

agent = CausalAgent(client, model="gpt-4o-mini")

# 도구 등록
agent.register_tool("get_data_columns", get_data_columns, test_tools[0])
agent.register_tool("compute_mean_diff", compute_mean_diff, test_tools[1])

# 자율 모드 실행
result = agent.run(
     "./dataset/online_classroom.csv 데이터에서 "
     "온라인 수업(format_ol)이 시험 점수(falsexam)에 미치는 효과를 분석해줘."
)

print("=== CausalAgent 테스트 준비 완료 ===")
print("agent.run() 또는 agent.run_pipeline()으로 실행할 수 있다.")


=== Step 1/10 ===
  [실행] get_data_columns

=== Step 2/10 ===
=== CausalAgent 테스트 준비 완료 ===
agent.run() 또는 agent.run_pipeline()으로 실행할 수 있다.


## 4. 메모리 관리

In [8]:
# ============================================================
# 메모리 관리: ConversationBufferMemory vs list[dict]
# ============================================================

# OpenAI SDK 방식: 단순한 리스트로 관리
memory = [
    {"role": "system", "content": "인과추론 에이전트"},
    {"role": "user", "content": "온라인 수업의 효과를 분석해줘"},
    {"role": "assistant", "content": "데이터를 먼저 분석하겠다."},
]

# 토큰 사용량 추정 (대략 4글자 = 1토큰)
def estimate_tokens(messages):
    """메시지 리스트의 토큰 수를 대략 추정한다."""
    total_chars = sum(len(m.get("content", "") or "") for m in messages)
    return total_chars // 4

print(f"현재 메모리 크기: {len(memory)}개 메시지")
print(f"추정 토큰 수: {estimate_tokens(memory)}개")

현재 메모리 크기: 3개 메시지
추정 토큰 수: 9개

→ LangChain의 ConversationBufferMemory 대신
→ 단순한 list[dict]로 동일한 기능을 구현할 수 있다.
